# Highly Pathogenic Avian Influenza (HPAI) Supervised Learning Models

This file contains our Scikit-learn supervised models, which can be compared to find the most effective model and to aid in writing the research manuscript. The Clean_Data.py must be ran inside this file so that the models have access to the data. There is plots, other figures, and accuracy scores so we can compare and validate our models. This has a couple of baseline models. Contributions by Brendan Daly.

In [1]:
import sys
# Load in libraries
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import GaussianNB
import pymc as pm
import pytensor.tensor as pt
import arviz as az

g++ not available, if using conda: `conda install gxx`


In [2]:
# PyMC needs Python 3.12 or 3.13 (not 3.14 yet); on Windows:
# pip install pandas openpyxl numpy scikit-learn pymc arviz
!{sys.executable} -m pip install pandas openpyxl numpy scikit-learn pymc arviz


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: C:\Users\Owner\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [3]:
# Run the cleaning pipeline once (takes ~1.5 min); skipped if outputs exist
from pathlib import Path
if not Path("data/clean/daily_milk.csv").exists():
    %run Clean_Data_KNN_Imputation.py
else:
    print("Cleaned data found -- skipping the pipeline")

Cleaned data found -- skipping the pipeline


In [5]:
# Load the cleaned tables produced by pipeline.py
daily_milk_data = pd.read_csv("data/clean/daily_milk.csv", parse_dates=["Date"])
hpai_labels = pd.read_csv("data/clean/hpai_labels.csv")

print(daily_milk_data.shape, hpai_labels.shape)

(12062, 17) (211, 5)


## Data Classification and Supervised Learning Setup

### Classify The Data

In [6]:
# Create a mapping from cow ID to binary ELISA classification
# (hpai_labels.csv already stores it as 0 = Negative, 1 = Suspect/Positive)
label_map = (
    hpai_labels.dropna(subset=["Cow_ID", "ELISA_Classification"])
    .set_index("Cow_ID")["ELISA_Classification"]
    .to_dict()
)

# Add the classification to every cow-day row
daily_milk_data["ELISA_Classification"] = (
    daily_milk_data["cow_ID"].astype("string").str.strip().map(label_map)
)

print(daily_milk_data["ELISA_Classification"].value_counts(dropna=False))

ELISA_Classification
0.0    6357
1.0    4765
NaN     940
Name: count, dtype: int64


### Create Dataframes for the Models to Use

In [7]:
# Keep only rows with known classifications
daily_milk_data = daily_milk_data.dropna(subset=["ELISA_Classification"])

# Features must be chosen explicitly
features = [
    "Day Production",
    "Milkings",
    "Refusals",
    "Failures",
]

X = daily_milk_data[features].apply(pd.to_numeric, errors="coerce")

# Binary label: 0 = Negative, 1 = Suspect/Positive
y = daily_milk_data["ELISA_Classification"].astype(int)

# Groups: one group per cow, so the split below never puts one cow's
# days in both train and test
groups = daily_milk_data["cow_ID"]

print(X.shape, "| label balance:", y.value_counts().to_dict())

(11122, 4) | label balance: {0: 6357, 1: 4765}


### Split the Data Up By Cow

In [8]:
# splitter prevents a single cow from being in both the test and training data sets due to blanket labelling inaccuracy
# 20% of data will be the test data + reproducibility
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

# Data separated based on cow ID rather than each row being shuffled
# X contains features, y contains classifications
train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Select features and their corresponding classifications
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

## Gaussian Naive Bayes Scikit-learn Model

In [13]:
Gauss_NB_Model = make_pipeline(
    SimpleImputer(strategy="median"),
    GaussianNB()
)

Gauss_NB_Model.fit(X_train, y_train)

y_pred = Gauss_NB_Model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.68      0.83      0.75      1572
           1       0.35      0.18      0.24       772

    accuracy                           0.62      2344
   macro avg       0.51      0.51      0.49      2344
weighted avg       0.57      0.62      0.58      2344



## Bayesian Logistic Regression PyMC Model

In [14]:
# Fit preprocessing only on the training set
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_processed = scaler.fit_transform(
    imputer.fit_transform(X_train)
)
X_test_processed = scaler.transform(
    imputer.transform(X_test)
)

with pm.Model() as logistic_model:

    intercept = pm.Normal("intercept", mu=0, sigma=1.5)

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=X_train_processed.shape[1]
    )

    logits = intercept + pt.dot(X_train_processed, beta)

    observed = pm.Bernoulli(
        "observed",
        logit_p=logits,
        observed=y_train.to_numpy().astype(int)
    )

    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=42
    )

az.summary(trace)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta]


C:\Users\Owner\AppData\Local\Programs\Python\Python314\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 15 seconds.


,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
intercept,-0.1822,0.0221,-0.22,-0.15,4621,2834,1.00,0.00032,0.00038
beta[0],-0.0823,0.0249,-0.12,-0.043,4002,2985,1.00,0.00039,0.0004
beta[1],-0.1477,0.0269,-0.19,-0.1,4032,3096,1.00,0.00042,0.00039
beta[2],0.0745,0.0248,0.035,0.11,4084,3098,1.00,0.00039,0.00039
beta[3],-0.0027,0.0217,-0.037,0.031,4534,2896,1.00,0.00032,0.00036
